# Data Overview


## NIST Photovoltaic Dataset

Dataset Source:  
https://pvdata.nist.gov/

---

## General Information

| Attribute | Description |
|---|---|
| **Location** | Maryland, United States |
| **Data Type** | Real-world photovoltaic system measurements |
| **Time Granularity** | 1-minute intervals |
| **Years Used** | 2015 until 2018 |

---

## Measurement Locations

| Location | Description |
|---|---|
| **Ground** | Solar panels installed on ground-mounted structures (fixed tilt ≈ 20°) |
| **Roof** | Solar panels installed on building rooftops |
| **Canopy** | Solar panels installed as parking canopy/shading structures |
| **ws_1, ws_2** | Independent weather stations collecting meteorological measurements |

---

## Dataset Size

| Dataset | Years Included | Total Rows |
|---|---|---|
| **Canopy Data** | 2015, 2016, 2017, 2018 | 2,102,400 |
| **Ground Data** | 2015, 2016, 2017, 2018 | 2,102,400 |
| **Roof Data** | 2015, 2016, 2017, 2018 | 1,706,400 |

## Colab setup

Run the cells in order in Google Colab. The first code cell clones the public GitHub repository into the Colab runtime and imports the shared helpers from `notebooks/Data Labeling/scripts/`. If the checkout already exists, it is reused; restart the runtime to clone a fresh copy after updating the repository.

The next cell mounts Google Drive. Store the raw daily CSVs under `MyDrive/NIST/`; merged CSVs are written to `MyDrive/NIST_processed/`. Only the code is cloned into the runtime. Data stays on Google Drive and is processed using Colab's memory.

The repository must be public before the clone cell can run without authentication.


In [1]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/dulkusim/pv-timeseries-fault-forecasting.git"
repo_path = Path("/content/pv-timeseries-fault-forecasting")

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(repo_path)],
        check=True,
    )

labeling_dir = repo_path / "notebooks" / "Data Labeling"
if not (labeling_dir / "scripts" / "io.py").is_file():
    raise FileNotFoundError(f"Shared scripts not found in {labeling_dir}")

if str(labeling_dir) not in sys.path:
    sys.path.insert(0, str(labeling_dir))

from scripts.io import get_data_root, count_daily_files, merge_daily_csvs


In [2]:
import os
from google.colab import drive

drive.mount("/content/drive")

# All raw data and merged outputs live on Google Drive.
os.environ["DATA_ROOT"] = "/content/drive/MyDrive"
root = get_data_root()
base_path = root / "NIST"
output_path = root / "NIST_Labeling"

if not base_path.is_dir():
    raise NotADirectoryError(f"Missing raw dataset directory: {base_path}")

dataset_dirs = sorted(path for path in base_path.iterdir() if path.is_dir())
if not dataset_dirs:
    raise ValueError(f"No dataset folders found in {base_path}")

print(f"Raw datasets: {base_path}")
print(f"Merged output: {output_path}")


Mounted at /content/drive
Raw datasets: /content/drive/MyDrive/NIST
Merged output: /content/drive/MyDrive/NIST_Labeling


## Dataset structure verification and file counting

The raw data is organized as `NIST/<dataset>/<month>/<daily file>.csv`, for example `NIST/onemin-Canopy-2015/01/`.

```text
NIST/
│
├── onemin-Canopy-2015/
│   ├── 01/
│   ├── 02/
│   └── ...
│
├── onemin-Canopy-2016/
│   ├── 01/
│   ├── 02/
│   └── ...
```

Each monthly folder contains daily CSV files, where:

- **1 CSV file = 1 day of measurements**
- Therefore, the number of files in a month should normally match the number of days in that month:
  - 31 files for months with 31 days
  - 30 files for months with 30 days
  - 28 or 29 files for February

Use the shared helper to count daily CSVs in every monthly directory. The nested `file_counts` dictionary is retained for further analysis. One file normally represents one day, so counts can be compared with the calendar month (including leap years).

This verification step is important because it helps identify:

- Missing files
- Incomplete downloads
- Corrupted or partially collected data
- Irregularities in the dataset structure


In [3]:
file_counts = {}

for folder_path in dataset_dirs:
    counts = count_daily_files(folder_path)
    file_counts[folder_path.name] = counts

    if not counts:
        print(f"{folder_path.name} | No monthly directories found")
    for month, count in counts.items():
        print(f"{folder_path.name} | Month {month} -> {count} files")


onemin-Canopy-2015 | Month 01 -> 31 files
onemin-Canopy-2015 | Month 02 -> 28 files
onemin-Canopy-2015 | Month 03 -> 31 files
onemin-Canopy-2015 | Month 04 -> 30 files
onemin-Canopy-2015 | Month 05 -> 31 files
onemin-Canopy-2015 | Month 06 -> 30 files
onemin-Canopy-2015 | Month 07 -> 31 files
onemin-Canopy-2015 | Month 08 -> 31 files
onemin-Canopy-2015 | Month 09 -> 30 files
onemin-Canopy-2015 | Month 10 -> 31 files
onemin-Canopy-2015 | Month 11 -> 30 files
onemin-Canopy-2015 | Month 12 -> 31 files
onemin-Canopy-2016 | Month 01 -> 31 files
onemin-Canopy-2016 | Month 02 -> 29 files
onemin-Canopy-2016 | Month 03 -> 31 files
onemin-Canopy-2016 | Month 04 -> 30 files
onemin-Canopy-2016 | Month 05 -> 31 files
onemin-Canopy-2016 | Month 06 -> 30 files
onemin-Canopy-2016 | Month 07 -> 31 files
onemin-Canopy-2016 | Month 08 -> 31 files
onemin-Canopy-2016 | Month 09 -> 30 files
onemin-Canopy-2016 | Month 10 -> 31 files
onemin-Canopy-2016 | Month 11 -> 30 files
onemin-Canopy-2016 | Month 12 -> 3

## Merging Daily CSV Files

This step combines all daily CSV files for each dataset folder into a single unified CSV file.

Use the shared merge helper once per dataset directory. It reads CSVs in sorted month and filename order, concatenates them, and saves `NIST_processed/<dataset>.csv` without a DataFrame index.

This preprocessing step simplifies later analysis and machine learning workflows by converting thousands of small daily files into a smaller number of continuous datasets.

The output also helps verify dataset completeness. Since the data has a 1-minute granularity:

- A normal year should contain:
  ```text
  24 × 60 × 365 = 525,600 records
  ```
- A leap year should contain:
  ```text
  24 × 60 × 366 = 527,040 records
  ```

The Canopy datasets match these expected values for 2015–2017, while the 2018 dataset contains fewer records, indicating missing data. Similar inconsistencies are also observed in some Ground and Roof datasets.


Memory is also managed efficiently by saving each merged dataset immediately and deleting it from memory after processing.

In [4]:
merged_paths = {}

for folder_path in dataset_dirs:
    saved_path = merge_daily_csvs(
        folder_path,
        output_path / f"{folder_path.name}.csv",
    )
    merged_paths[folder_path.name] = saved_path
    print(f"Saved: {saved_path}")


Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Canopy-2015.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Canopy-2016.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Canopy-2017.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Canopy-2018.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Ground-2015.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Ground-2016.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Ground-2017.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Ground-2018.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Roof-2015.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Roof-2016.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Roof-2017.csv
Saved: /content/drive/MyDrive/NIST_Labeling/onemin-Roof-2018.csv
